In [1]:
# Abstraction Show Case - Complete Payment Module - Project

from abc import ABC, abstractmethod
import uuid
import time
import random

# =========================
# 1) Abstraction Layer (Interface)
# =========================

class PaymentGateway(ABC):
    """Abstract base class: defines the contract for all payment methods."""
    @abstractmethod
    def pay(self, amount: float) -> str:
        """Process a payment and return a user-visible message."""
        pass


# =========================
# 2) Hidden Internal Services (Implementation Details)
# =========================

class UPIValidator:
    """Validates UPI IDs and basic input constraints."""
    def validate_upi_id(self, upi_id: str) -> bool:
        # Rule: must contain '@' and provider suffix, e.g., 'name@bank'
        if not upi_id or '@' not in upi_id:
            return False
        name, provider = upi_id.split('@', maxsplit=1)
        if not name or not provider:
            return False
        # Simulate provider whitelist
        allowed = {"okicici", "oksbi", "okaxis", "okhdfcbank", "paytm", "ybl"}
        return provider in allowed

    def validate_amount(self, amount: float) -> bool:
        # Rule: positive and within a reasonable limit
        return amount > 0 and amount <= 100000


class CryptoService:
    """Encrypts sensitive payloads."""
    def encrypt(self, payload: dict) -> str:
        # Simulate encryption by producing a tokenized string
        token = uuid.uuid4().hex
        return f"enc::{token}::{payload.get('upi_id','unknown')}::{payload.get('amount','0')}"


class BankAPIClient:
    """Communicates with the bank server."""
    def initiate_transaction(self, encrypted_payload: str) -> dict:
        # Simulate network latency
        time.sleep(0.2)
        # Simulate random outcomes
        outcome = random.choice(["success", "insufficient_funds", "network_error"])
        txn_id = uuid.uuid4().hex[:12]
        if outcome == "success":
            return {"status": "success", "transaction_id": txn_id}
        elif outcome == "insufficient_funds":
            return {"status": "failed", "reason": "Insufficient funds"}
        else:
            return {"status": "failed", "reason": "Network error"}

    def verify_auth(self, method: str, value: str) -> bool:
        # Simulate PIN/OTP verification
        if method == "PIN":
            return value == "1234"  # demo PIN
        if method == "OTP":
            return len(value) == 6 and value.isdigit()
        return False


class AuditLogger:
    """Logs transactions for audit and compliance."""
    def log(self, event: str, details: dict) -> None:
        # In real systems, this would write to a secure log store
        print(f"[AUDIT] {event} :: {details}")


class ErrorHandler:
    """Centralized error handling and user-friendly messaging."""
    def to_user_message(self, error: str) -> str:
        mapping = {
            "Insufficient funds": "Payment failed: insufficient balance.",
            "Network error": "Payment failed due to network issues. Please try again.",
            "Invalid UPI ID": "Payment failed: invalid UPI ID.",
            "Invalid amount": "Payment failed: invalid amount.",
            "Auth failed": "Payment failed: authentication failed.",
        }
        return mapping.get(error, "Payment failed due to an unknown error.")


# =========================
# 3) Concrete Payment Method (Implements the Abstraction)
# =========================

class UPIPayment(PaymentGateway):
    """Concrete implementation of PaymentGateway for UPI."""
    def __init__(self, upi_id: str, auth_method: str = "PIN", auth_value: str = "1234"):
        self.upi_id = upi_id
        self.auth_method = auth_method
        self.auth_value = auth_value

        # Hidden internal services
        self.validator = UPIValidator()
        self.crypto = CryptoService()
        self.bank = BankAPIClient()
        self.audit = AuditLogger()
        self.errors = ErrorHandler()

    def pay(self, amount: float) -> str:
        # --- Hidden Internal Details ---
        # Step 1: Validate inputs
        if not self.validator.validate_upi_id(self.upi_id):
            self.audit.log("VALIDATION_FAILED", {"upi_id": self.upi_id})
            return self.errors.to_user_message("Invalid UPI ID")

        if not self.validator.validate_amount(amount):
            self.audit.log("VALIDATION_FAILED", {"amount": amount})
            return self.errors.to_user_message("Invalid amount")

        # Step 2: Authenticate user (PIN/OTP)
        if not self.bank.verify_auth(self.auth_method, self.auth_value):
            self.audit.log("AUTH_FAILED", {"method": self.auth_method})
            return self.errors.to_user_message("Auth failed")

        # Step 3: Encrypt payload
        payload = {"upi_id": self.upi_id, "amount": amount}
        encrypted = self.crypto.encrypt(payload)
        self.audit.log("ENCRYPTED", {"encrypted": encrypted[:30] + "..."} )

        # Step 4: Connect to bank server and process transaction
        response = self.bank.initiate_transaction(encrypted)
        self.audit.log("BANK_RESPONSE", response)

        # Step 5: Handle outcomes and errors
        if response.get("status") == "success":
            txn_id = response.get("transaction_id")
            self.audit.log("PAYMENT_SUCCESS", {"txn_id": txn_id, "amount": amount})
            # --- Visible Output to User ---
            return f"Paid {amount} using UPI (Txn: {txn_id})"
        else:
            reason = response.get("reason", "Unknown")
            self.audit.log("PAYMENT_FAILED", {"reason": reason})
            return self.errors.to_user_message(reason)


# =========================
# 4) Orchestration (User-Facing)
# =========================

def checkout(payment_method: PaymentGateway, amount: float) -> None:
    """User-facing function: calls the abstract interface without knowing internals."""
    message = payment_method.pay(amount)
    print(message)


# =========================
# 5) Demo Usage
# =========================

if __name__ == "__main__":
    # Valid UPI ID, correct PIN
    upi = UPIPayment(upi_id="ananda@okhdfcbank", auth_method="PIN", auth_value="1234")
    checkout(upi, 500.0)

    # Invalid UPI ID
    upi_bad = UPIPayment(upi_id="ananda@", auth_method="PIN", auth_value="1234")
    checkout(upi_bad, 300.0)

    # Wrong PIN
    upi_wrong_pin = UPIPayment(upi_id="ananda@okaxis", auth_method="PIN", auth_value="9999")
    checkout(upi_wrong_pin, 200.0)

    # OTP flow
    upi_otp = UPIPayment(upi_id="ananda@ybl", auth_method="OTP", auth_value="654321")
    checkout(upi_otp, 750.0)


[AUDIT] ENCRYPTED :: {'encrypted': 'enc::464a3859bc734208987a78081...'}
[AUDIT] BANK_RESPONSE :: {'status': 'failed', 'reason': 'Network error'}
[AUDIT] PAYMENT_FAILED :: {'reason': 'Network error'}
Payment failed due to network issues. Please try again.
[AUDIT] VALIDATION_FAILED :: {'upi_id': 'ananda@'}
Payment failed: invalid UPI ID.
[AUDIT] AUTH_FAILED :: {'method': 'PIN'}
Payment failed: authentication failed.
[AUDIT] ENCRYPTED :: {'encrypted': 'enc::2fe07f649aa2436e928b9d594...'}
[AUDIT] BANK_RESPONSE :: {'status': 'success', 'transaction_id': 'e19a71f949eb'}
[AUDIT] PAYMENT_SUCCESS :: {'txn_id': 'e19a71f949eb', 'amount': 750.0}
Paid 750.0 using UPI (Txn: e19a71f949eb)


In [3]:
# Above Code works as Where abstraction happens

# Abstract class (PaymentGateway)
# Role: Defines the contract—every payment method must implement pay(amount).
# Abstraction: The caller (e.g., checkout) only knows it can call pay; it doesn’t know or care how UPI,
#  Credit Card, or PayPal implement it.

# Abstract method (pay)
# Role: Specifies “what” must be done (process payment) without dictating “how.”
# Abstraction: The internal steps—validation, encryption, bank API, auth, logging—are 
# hidden behind this single method call.

# Orchestration (checkout)
# Role: Works with the abstract type PaymentGateway, not concrete classes.
# Abstraction: checkout is decoupled from the implementation details; it can accept any 
# payment type that conforms to the interface.

# How it works (execution flow)
# User triggers payment via checkout(UPIPayment(...), amount).
# checkout calls pay(amount) on the abstract interface—no knowledge of internals.

# Inside UPIPayment.pay:
# Validate UPI ID and amount.
# Authenticate via PIN/OTP.
# Encrypt payload.
# Call bank API and process response.
# Log audit events.
# Return user-friendly message (success or failure).

# User sees only the final message printed by checkout.

# What the end user sees vs. what is hidden
# Visible to end user:
# Success: Paid 500.0 using UPI (Txn: ab12cd34ef56)
# Failure: Payment failed: insufficient balance. or Payment failed due to network issues. Please try again.

# Hidden (abstracted away):
# Validation: UPI ID format, amount limits.
# Security: Encryption of payload.
# Auth: PIN/OTP verification.
# Integration: Bank API calls, network latency, response parsing.
# Error handling: Mapping technical errors to user-friendly messages.
# Audit: Structured logging of events and outcomes.

# Core logic to understand abstraction
# Define a contract with an abstract class—focus on “what” (process payment).
# Hide complexity inside concrete implementations—focus on “how” (validation, encryption, API).
# Program to an interface (PaymentGateway)—callers don’t depend on specific implementations.
# Swap implementations easily—add CreditCardPayment, PayPalPayment, etc., without changing checkout.
# If you want, I can add a CreditCardPayment and PayPalPayment class to show how the same abstraction 
# supports multiple payment methods with different internals.

In [2]:
# Link
# https://chatgpt.com/share/6961145a-ec78-8002-83e1-68a5a6d6b33a

# Code Explanation

class Bank_Account:
# class → keyword used to define a class (blueprint)
# Bank_Account → name of the class
# : → starts the class body

    MIN_BALANCE = 1000
    # MIN_BALANCE → class variable (shared by all objects)
    # = → assignment operator
    # 1000 → minimum balance value
    # class-level constant → does not belong to a single object

    def __init__(self, owner, balance=0):
    # def → keyword to define a function/method
    # __init__ → constructor (runs automatically when object is created)
    # self → reference to current object
    # owner → parameter to store account holder name
    # balance=0 → default argument (if not passed, value is 0)

        self.owner = owner
        # self → current object
        # .owner → instance variable
        # owner → value passed during object creation

        self.balance = balance
        # self.balance → instance variable storing account balance
        # balance → value received from parameter

    def deposit(self, amount):
    # deposit → instance method name
    # amount → money to be deposited

        if amount <= 0:
        # if → conditional keyword
        # amount → deposit value
        # <= → less than or equal operator
        # checks: is deposit invalid?

            print("Deposit amount must be positive")
            # print → displays error message

        else:
        # else → runs when if condition is false

            self.balance += amount
            # += → addition assignment
            # means: self.balance = self.balance + amount

            print(f"{amount} deposited. New balance is {self.balance}")
            # f-string → formatted output
            # {amount} → deposit value
            # {self.balance} → updated balance

    def withdraw(self, amount):
    # withdraw → method to remove money
    # amount → withdrawal value

        if amount <= 0:
        # checks if withdrawal is invalid

            print("Withdrawal amount must be positive")
            # prints error message

        elif self.balance - amount < Bank_Account.MIN_BALANCE:
        # elif → else if condition
        # self.balance → current balance
        # - → subtraction
        # Bank_Account.MIN_BALANCE → access class variable
        # checks if withdrawal breaks minimum balance rule

            print("Withdrawal denied. Minimum balance of 1000 must be maintained")
            # prints minimum balance warning

        else:
        # runs when all validations pass

            self.balance -= amount
            # -= → subtraction assignment
            # means: self.balance = self.balance - amount

            print(f"{amount} withdrawn. Balance is {self.balance}")
            # prints successful withdrawal message

    def get_balance(self):
    # get_balance → method to fetch balance

        return self.balance
        # return → sends value back to caller
        # self.balance → current balance value


# Creating an object (instance) of the class
account = Bank_Account("Anand", 5000)
# account → object reference variable
# Bank_Account → class name
# ("Anand", 5000) → arguments passed to constructor

account.deposit(-100)
# calls deposit method
# -100 → invalid deposit

account.deposit(500)
# valid deposit
# balance becomes 5500

account.withdraw(4500)
# invalid withdrawal (breaks minimum balance rule)

account.withdraw(2000)
# valid withdrawal
# balance becomes 3500

print(account.get_balance())
# get_balance() → returns final balance
# print → displays final output



Deposit amount must be positive
500 deposited. New balance is 5500
4500 withdrawn. Balance is 1000
Withdrawal denied. Minimum balance of 1000 must be maintained
1000


Deposit amount must be positive
500 deposited. New balance is 5500
4500 withdrawn. Balance is 1000
Withdrawal denied. Minimum balance of 1000 must be maintained
1000
